# Lab 2: Test & Iterate

**Workshop 1, block 5. 25 minutes.**

Every team completes at least one full cycle and reports a result.

1. Index the sample corpus and answer the 15 questions
2. Change exactly one setting
3. Rebuild if you changed chunking, then re-run
4. Mark the answers and write the row into your log
5. Decide whether to keep it, then go again

**Change one setting at a time.** Change chunk size and k together and a
rising score tells you nothing about which helped. The two often pull in
opposite directions.

> **Before you start:** `sample_corpus/` and `dev_set.json`. Your k=5 number from lab 1.
>
> **When you finish:** `data/chroma` (collection `workshop`) and `experiment_log.csv`. **Labs 3, 4 and 5 all read that index, so run this before Workshop 2.**

---
## Setup

Run these two cells first. They are identical in every lab, so each
notebook works on its own.

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops. **Do not commit a notebook with a key
visible in its output.**

In [ ]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

In [ ]:
# ---- helpers ------------------------------------------------------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# One PersistentClient per path, cached for the life of this kernel.
# chromadb caches internal state per path, so deleting the folder and
# opening a fresh PersistentClient while an earlier one from this same
# session is still alive corrupts the connection: you get "attempt to
# write a readonly database" or "database is locked" on the very next
# call. Rebuilding your index more than once per session, which the
# "change one setting, re-run" loop asks you to do, hits this every
# time with the naive version.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P

    if path not in _stores:
        # First time this path is opened in this session. Safe to wipe a
        # stale, wrong-chromadb-version index here, since no client for
        # this path exists in this process yet.
        if reset and _P(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by "
                f"a different chromadb version. Delete that folder and rebuild, or "
                f"install the pinned version from requirements.txt."
            ) from None

    client = _stores[path]
    if reset:
        # Reset now means delete-and-recreate the COLLECTION on the same
        # client, not delete-and-recreate the DIRECTORY under it. This is
        # what actually avoids the readonly/locked error on every rebuild
        # after the first.
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT, max_tokens=800,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

---
## The pipeline

Everything you change lives in `CONFIG`. Nothing else needs editing.

In [ ]:
CONFIG = {
    "chunk_size": 800,    # try 400 to 1200
    "overlap":    100,    # try 0 to 20% of chunk size
    "k":          5,      # try 3 to 10
}

SYSTEM_PROMPT = """You answer questions about the Tam Wing Fan Innovation Wing.

Answer only from the context below. Where the context disagrees with what
you think you know, the context is correct.

Reply with the answer only. No explanation, no preamble. If the question
asks how many, reply with a number.

If the context does not contain the answer, give your best guess anyway.
Never reply that you do not know."""


def build_index(config, corpus_dir="sample_corpus", reset=True):
    texts, metas = [], []
    for f in sorted(Path(corpus_dir).glob("*.txt")):
        for i, piece in enumerate(chunk(f.read_text(),
                                        config["chunk_size"], config["overlap"])):
            texts.append(piece)
            metas.append({"url": f.name, "position": i, "kind": "text"})
    store = get_store("data/chroma", name="workshop", reset=reset)
    add_to_store(store, texts, metas)
    print(f"indexed {len(texts)} chunks "
          f"(size={config['chunk_size']}, overlap={config['overlap']})")
    return store


def answer_one(store, question, config):
    chunks  = query(store, question, k=config["k"])
    context = "\n\n".join(f"[{c['metadata'].get('url','?')}]\n{c['text']}"
                           for c in chunks)
    reply = chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])
    return reply.strip(), chunks

---
## Step 1: Baseline

Run it, then **read the fifteen answers**. Mark each yourself: 1
correct, 0.5 partly, 0 wrong. It takes about three minutes, and reading
them is how you notice the bot answered a slightly different question,
hedged, or returned three paragraphs where one word was wanted.

In [ ]:
def run_dev_set(store, dev, config):
    rows, times = [], []
    for item in dev:
        (reply, chunks), secs = timed(answer_one, store, item["question"], config)
        times.append(secs)
        rows.append({**item, "given": reply, "chunks": chunks, "seconds": secs})
    print(f"median {statistics.median(times):.1f}s   slowest {max(times):.1f}s")
    if max(times) > 30:
        print("WARNING: at least one question is over the 30 second limit.")
    return rows


def review(rows):
    for r in rows:
        got = "yes" if answer_present(r["answer"], r["chunks"]) else "NO"
        print(f"Lv{r['level']}  {r['question']}")
        print(f"   expected:  {r['answer']}")
        print(f"   got:       {r['given'] or '(empty)'}")
        print(f"   retrieved: {got}    {r['seconds']:.1f}s\n")


def log_run(row, path="experiment_log.csv"):
    import csv
    p = Path(path); new = not p.exists()
    with p.open("a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(row.keys()))
        if new:
            w.writeheader()
        w.writerow(row)
    print("appended to", p)


dev   = load_dev_set("dev_set.json")
store = build_index(CONFIG)
rows  = run_dev_set(store, dev, CONFIG)
review(rows)

In [ ]:
log_run({
    "note":      "baseline",
    "chunk":     CONFIG["chunk_size"],
    "overlap":   CONFIG["overlap"],
    "k":         CONFIG["k"],
    "retrieved": sum(answer_present(r["answer"], r["chunks"]) for r in rows),
    "correct":   0,                     # <-- your marks go here
    "slowest":   round(max(r["seconds"] for r in rows), 1),
})

---
## Steps 2 to 4: Change one setting, re-run, log it

**Rebuild only if you changed chunk_size or overlap.** Those change the
stored data. Changing k or the prompt does not.

In [ ]:
CONFIG["chunk_size"] = 500            # change ONE, leave the rest

store = build_index(CONFIG)           # rebuild: chunking changed
rows  = run_dev_set(store, dev, CONFIG)
review(rows)

log_run({
    "note":      "chunk 800 -> 500",
    "chunk":     CONFIG["chunk_size"],
    "overlap":   CONFIG["overlap"],
    "k":         CONFIG["k"],
    "retrieved": sum(answer_present(r["answer"], r["chunks"]) for r in rows),
    "correct":   0,                     # <-- your marks go here
    "slowest":   round(max(r["seconds"] for r in rows), 1),
})

---
## Reading your log

| What you see | What it means |
|---|---|
| Retrieved rising, correct flat | The prompt is now the limit |
| Both rising | Keep the change |
| Slowest rising sharply | Warning. 30 seconds is a hard limit |
| Neither moving | That setting was not the bottleneck |

Append every run. By mid-October you will want to know whether 800 with
100 overlap beat 600 with 50, and nobody remembers.

---
## Going further

Sweep k, then sweep chunk size at your best k. The two interact, which
is why you change one at a time.

In [ ]:
for k in (3, 5, 8, 10, 15):
    CONFIG["k"] = k
    rows_k = run_dev_set(store, dev, CONFIG)
    got = sum(answer_present(r["answer"], r["chunks"]) for r in rows_k)
    print(f"  k={k:<3} retrieved {got}/{len(dev)}\n")

---
## Before you close this notebook

Your index is on disk at `data/chroma`, collection `workshop`. Labs 3, 4
and 5 all open it. Do not delete it.